In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
#CNN imports 
# mobileNetV2 and EfficientNetB0 
import tensorflow as tf
import numpy as np
#type: ignore added to ignore type checking for these imports code works fine without type checking
from tensorflow.keras.applications import MobileNetV2 # type: ignore
from tensorflow.keras.preprocessing.image import ImageDataGenerator # type: ignore
from tensorflow.keras.models import Model # type: ignore
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout # type: ignore
from tensorflow.keras.optimizers import Adam # type: ignore
from tensorflow.keras.utils import Sequence # type: ignore
from PIL import Image




In [2]:
df_DE = pd.read_csv("DE_Img.csv")


#will split to 70% and temp 30%
train_df, temp_df = train_test_split(df_DE, test_size=0.30, random_state=50, shuffle=True)
#will split temp to 50% for validation and 50% for test
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=50, shuffle=True)

print("Train set size:", len(train_df), "Validation set size:", len(val_df),
      "Test set size:", len(test_df))

Train set size: 905 Validation set size: 194 Test set size: 195


In [4]:
df_SD = pd.read_csv("SD_Img.csv")


#will split to 70% and temp 30%
train_df, temp_df = train_test_split(df_SD, test_size=0.30, random_state=50, shuffle=True)
#will split temp to 50% for validation and 50% for test
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=50, shuffle=True)

print("Train set size:", len(train_df), "Validation set size:", len(val_df),
      "Test set size:", len(test_df))

Train set size: 905 Validation set size: 194 Test set size: 195


In [6]:
df_H = pd.read_csv("HP_Img.csv")


#will split to 70% and temp 30%
train_df, temp_df = train_test_split(df_H, test_size=0.30, random_state=50, shuffle=True)
#will split temp to 50% for validation and 50% for test
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=50, shuffle=True)

print("Train set size:", len(train_df), "Validation set size:", len(val_df),
      "Test set size:", len(test_df))

Train set size: 905 Validation set size: 194 Test set size: 195


In [8]:
df_C = pd.read_csv("Complete_Img.csv")


#will split to 70% and temp 30%
train_df, temp_df = train_test_split(df_C, test_size=0.30, random_state=50, shuffle=True)
#will split temp to 50% for validation and 50% for test
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=50, shuffle=True)

print("Train set size:", len(train_df), "Validation set size:", len(val_df),
      "Test set size:", len(test_df))




Train set size: 2717 Validation set size: 582 Test set size: 583


In [9]:
# ----------------------------MobileNetV2 Model Training Setup--------------------------------#
#we use MobileNetV2 as the base model, which is a lightweight model suitable for image classification tasks
# MobileNetV2 is pre-trained on ImageNet, which helps in transfer learning, transfer learning means we will use the weights from the pre-trained model and fine-tune it on our dataset
#this is good for our case because we have a limited dataset and we want to leverage the knowledge from the pre-trained model
#movileNetV2 expects images of size 224x224, so we will resize our images to this size
# We will also create a custom data generator to handle image loading, resizing, and label encoding


#because some images may be corrupted or not readable, we will create a custom generator that handles these cases gracefully
# This generator will skip any images that cannot be loaded and will not crash the training process.
class SafeImageDataGenerator(Sequence):
    #initialization method for the generator
    # dataframe: pandas DataFrame containing image paths and labels, class_indices: dictionary mapping class names to indices
    def __init__(self, dataframe, x_col, y_col, batch_size, target_size, class_indices, shuffle=True):
        self.df = dataframe.reset_index(drop=True) # Reset index to ensure consistent indexing
        self.x_col = x_col # Column name for image paths
        self.y_col = y_col # Column name for labels
        self.batch_size = batch_size # Size of each batch
        self.target_size = target_size # Target size for resizing images
        self.class_indices = class_indices # Dictionary mapping class names to indices
        self.shuffle = shuffle # Whether to shuffle the data at the start of each epoch
        self.indices = np.arange(len(self.df)) # Create an array of indices for the DataFrame
        
        # Shuffle indices 
        if self.shuffle:
            np.random.shuffle(self.indices) # Shuffle indices to randomize the order of data

    # len method to return the number of batches per epoch
    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))# Calculate the number of batches per epoch

    #getitem method to retrieve a batch of data, idx: index of the batch to retrieve
    def __getitem__(self, idx):
        #batch_indices: indices for the current batch, this is used to slice the DataFrame
        batch_indices = self.indices[idx * self.batch_size : (idx + 1) * self.batch_size]
        batch_x = [] # List to hold images for the batch
        batch_y = [] # List to hold labels for the batch

        # Loop through each index in the batch_indices to load images and labels
        for i in batch_indices:
            row = self.df.iloc[i] # Get the row corresponding to the current index
            img_path = row[self.x_col] # Get the image path from the DataFrame
            label_str = str(row[self.y_col]) # Get the label from the DataFrame and convert it to string

            # Check if the label exists in class_indices, if not, skip this image
            if label_str not in self.class_indices:
                # If label is not found in class_indices, skip this image, 
                print(f"Warning: Label '{label_str}' not found in class indices, skipping.")
                continue 
            label_index = self.class_indices[label_str] # Get the index of the label from class_indices
            #one-hot means we will create a vector where the index corresponding to the label is set to 1 and all others are 0
            # This is useful for multi-class classification problems, thought we have binary classification here, 
            # we still wanted to use one-hot encoding for consistency and learning for future models
            label_one_hot = np.zeros(len(self.class_indices)) # Create a one-hot encoded vector for the label
            label_one_hot[label_index] = 1 # Set the index corresponding to the label to 1
            # Try to open the image, resize it, and convert it to a numpy array, need to handle any exceptions that may occur
            
            try:
                img = Image.open(img_path).convert('RGB') # Open the image and convert it to RGB format
                img = img.resize(self.target_size) # Resize the image to the target size
                img_array = np.array(img) / 255.0 # Normalize the image array to [0, 1] because pixel values are usually in range [0, 255]
                batch_x.append(img_array)# Append the image array to the batch_x list
                batch_y.append(label_one_hot) # Append the one-hot encoded label to the batch_y list
            # If an error occurs while opening or processing the image, skip this image
            except Exception as e:
                print(f"Skipping bad image: {img_path} | Error: {e}")
                continue
        # If no valid images were added to the batch, return a dummy batch, needed to avoid errors during training
        if len(batch_x) == 0:
            # Return dummy batch if everything in this batch failed
            dummy_x = np.zeros((1, *self.target_size, 3), dtype=np.float32)# Create a dummy image with the target size and 3 channels (RGB)
            dummy_y = np.zeros((1, len(self.class_indices)), dtype=np.float32)# Create a dummy label with the same number of classes
            return dummy_x, dummy_y # Return the dummy batch
        # Convert the lists to numpy arrays and return them as a batch, needed for training because Keras expects numpy arrays
        return np.array(batch_x, dtype=np.float32), np.array(batch_y, dtype=np.float32)
    # on_epoch_end method to shuffle the indices at the end of each epoch if shuffle is True this allows the model to see the data in a different order each epoch
    #helpful for training to avoid overfitting and to ensure that the model learns from different samples in each epoch
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices) #randomly shuffle the indices to change the order of the data for the next epoch

##-----------------------------MobileNetV2 Model Parameters--------------------------------#

IMG_SIZE = (224, 224) # Target size for resizing images, MobileNetV2 expects 224x224 images
BATCH_SIZE = 32 # Batch size for training, can be adjusted based on available memory and GPU capabilities, we chose 32 as a common default

# Convert labels to string type for consistency, this is important because the class_indices dictionary uses string labels
# and we want to ensure that the labels in the DataFrame match the keys in class_indices
for df in [train_df, val_df, test_df]: #loops through each DataFrame
    df['label'] = df['label'].astype(str) #will convert the 'label' column to string type

# will create a mapping from class names to indices, this is useful for converting string labels to numerical indices for training
unique_labels = sorted(train_df['label'].unique()) # # Get unique labels from the training DataFrame and sort them
class_indices = {label: idx for idx, label in enumerate(unique_labels)} # # Create a dictionary mapping each label to its index

print(f"Classes: {class_indices}") # Print the class indices for reference, this will help us understand how the labels are mapped to indices

##------------------------------Data Generators--------------------------------##
#data generators for training, validation, and test sets
# These generators will handle loading images, resizing them, and converting labels to one-hot encoded vectors
#we need this to ensure that the model can learn from the images and labels correctly

# We will use the SafeImageDataGenerator class to handle any issues with image loading
#training generator, we will shuffle the training data to ensure that the model sees the data in a different order each epoch
train_gen = SafeImageDataGenerator(
    dataframe=train_df, 
    x_col='image',
    y_col='label',
    batch_size=BATCH_SIZE,
    target_size=IMG_SIZE,
    class_indices=class_indices,
    shuffle=True
)

# Validation generator, we will not shuffle the validation data to maintain the order
val_gen = SafeImageDataGenerator(
    dataframe=val_df,
    x_col='image',
    y_col='label',
    batch_size=BATCH_SIZE,
    target_size=IMG_SIZE,
    class_indices=class_indices,
    shuffle=False
)


##------------------------------Model Definition--------------------------------##
# We will use MobileNetV2 as the base model, which is a lightweight model suitable for image classification tasks
# MobileNetV2 is pre-trained on ImageNet, which helps in transfer learning

# This means we will use the weights from the pre-trained model and fine-tune it on our dataset
#input_shape = means the shape of the input images, we will use 224x224, include_top=False means we will not include the fully connected layers at the top of the model
#weights control whether to use pre-trained weights. these weights measure the performance of the model on ImageNet dataset
base_model = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False  # Freeze pretrained base

# We will add our own classification head on top of the base model
x = base_model.output # Add the output of the base model
x = GlobalAveragePooling2D()(x) # Add global average pooling to reduce the spatial dimensions, this helps in reducing the number of parameters and prevents overfitting
x = Dropout(0.3)(x) # Add dropout layer to reduce overfitting, this randomly sets a fraction of input units to 0 at each update during training time
# This helps in preventing overfitting by introducing noise during training
# Add a dense layer with softmax activation for multi-class classification, this layer will output the probabilities for each class
output = Dense(len(class_indices), activation='softmax')(x) 

# Create the final model by combining the base model and the classification head
# The inputs are the input of the base model and the outputs are the output of the classification
model = Model(inputs=base_model.input, outputs=output)
# Compile the model with Adam optimizer and categorical crossentropy loss, this is suitable for multi-class classification problems
#adam optimizer will adjust the learning rate during training, categorical crossentropy is used for multi-class classification problems
#categorical_crossentropy is used because we have multiple classes and we want to minimize the difference between the predicted probabilities and the true labels
model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()# Print the model summary to see the architecture and number of parameters

##------------------------------Model Training--------------------------------##
# We will train the model using the training generator and validate it using the validation generator

EPOCHS = 10 # Number of epochs to train the model, epochs are the number of times the model will see the entire training dataset
#high epochs can lead to overfitting, for 3,8882 images, 10 epochs is a reasonable starting point

#history: will store the training history, this will contain the loss and accuracy for each epoch
# We will use the fit method to train the model, passing in the training and validation generators
#model.fit() will train the model for a fixed number of epochs
history = model.fit(
    train_gen, # Training generator
    validation_data=val_gen, # Validation generator
    epochs=EPOCHS # Number of epochs to train the model
)

##------------------------------Model Evaluation--------------------------------##
# After training, we will evaluate the model on the test set to see how well it performs
# We will use the SafeImageDataGenerator for the test set as well, but we will not shuffle the data
# This is important because we want to evaluate the model on the same order of images as in the test set
# This will help us understand how well the model generalizes to unseen data

# Test generator, we will not shuffle the test data to maintain the order
test_gen = SafeImageDataGenerator(
    dataframe=test_df,
    x_col='image',
    y_col='label',
    batch_size=1,   # batch size 1 for precise evaluation
    target_size=IMG_SIZE,
    class_indices=class_indices,
    shuffle=False
)

##-------------------------------Model Prediction--------------------------------##
# We will use the model to predict the labels for the test set
#this will give us the predicted probabilities for each class, needed to evaluate the model performance
# We will store the true labels and predicted labels for evaluation
y_true = [] # List to store true labels
y_pred = [] # List to store predicted labels

# Loop through the test generator to get predictions for each batch
# We will use the model.predict() method to get the predictions for each batch
for i in range(len(test_gen)):
    x_batch, y_batch = test_gen[i] # Get the batch of images and labels
    # If the batch is empty, skip to the next iteration
    if x_batch.shape[0] == 0:
        continue
    # Predict the labels for the batch using the model
    preds = model.predict(x_batch)
    y_true.append(np.argmax(y_batch, axis=1)[0]) # Get the true label for the batch
    y_pred.append(np.argmax(preds, axis=1)[0]) # Get the predicted label for the batch

#print the classification report to see the precision, recall, and F1-score for each class
print(classification_report(y_true, y_pred, target_names=unique_labels, digits=4))

Classes: {'Hope_speech': 0, 'Non_hope_speech': 1}


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_3[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,260,546 (8.62 MB)

 Trainable params: 2,562 (10.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

C:\Users\nutme\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
C:\Users\nutme\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\PIL\Image.py:3570: UserWarning: image file could not be identified because AVIF support not installed
  warnings.warn(message)


Skipping bad image: images/IHS_0542.jpg | Error: cannot identify image file 'images/IHS_0542.jpg'
Epoch 1/10
Skipping bad image: images/IHS_0542.jpg | Error: cannot identify image file 'images/IHS_0542.jpg'
 2/85 ━━━━━━━━━━━━━━━━━━━━ 31s 383ms/step - accuracy: 0.5593 - loss: 0.8941Skipping bad image: images/IHS_0381.jpg | Error: cannot identify image file 'images/IHS_0381.jpg'
18/85 ━━━━━━━━━━━━━━━━━━━━ 31s 476ms/step - accuracy: 0.5289 - loss: 0.8916Skipping bad image: images/IHS_0020.jpg | Error: cannot identify image file 'images/IHS_0020.jpg'
21/85 ━━━━━━━━━━━━━━━━━━━━ 30s 469ms/step - accuracy: 0.5279 - loss: 0.8891Skipping bad image: images/IHS_0130.jpg | Error: cannot identify image file 'images/IHS_0130.jpg'
27/85 ━━━━━━━━━━━━━━━━━━━━ 27s 470ms/step - accuracy: 0.5276 - loss: 0.8831Skipping bad image: images/IHS_0444.jpg | Error: cannot identify image file 'images/IHS_0444.jpg'
34/85 ━━━━━━━━━━━━━━━━━━━━ 23s 456ms/step - accuracy: 0.5295 - loss: 0.8753Skipping bad image: images